In [ ]:
import os
import operator
import math
import datetime
from typing import TypedDict, Annotated, Literal
from dotenv import load_dotenv

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import (
    HumanMessage, AIMessage, SystemMessage, ToolMessage, BaseMessage,
)
from langchain_core.tools import tool
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver

load_dotenv(override=True)
gemini_api_key = os.getenv("GEMINI_API_KEY")
if not gemini_api_key:
    raise ValueError("GEMINI_API_KEY not found. Add it to your .env file.")

chat_model = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.3,
    google_api_key=gemini_api_key,
)

In [ ]:
# ── Cyclic Agent Graph: LLM \u2194 Tools Loop ────────────────────────────────
print("\U0001f504 PART B \u2013 Loops: Cyclic Agent Graph")
print("=" * 55)

# ── Tool definitions ──────────────────────────────────────────────────────
@tool
def add_numbers(a: float, b: float) -> float:
    """Add two numbers together and return the result."""
    return a + b

@tool
def get_current_date() -> str:
    """Return today's date in YYYY-MM-DD format."""
    return datetime.date.today().isoformat()

@tool
def calculate_square_root(number: float) -> float:
    """Calculate the square root of a non-negative number."""
    if number < 0:
        return "Error: cannot compute square root of a negative number."
    return round(math.sqrt(number), 4)

agent_tools    = [add_numbers, get_current_date, calculate_square_root]
tool_map       = {t.name: t for t in agent_tools}
llm_with_tools = chat_model.bind_tools(agent_tools)

# ── State: messages ACCUMULATE via operator.add ───────────────────────────
class AgentState(TypedDict):
    messages: Annotated[list[BaseMessage], operator.add]

# ── Nodes ─────────────────────────────────────────────────────────────────
def agent_llm_node(state: AgentState) -> dict:
    """Ask the LLM what to do next. Returns a tool call or a final answer."""
    return {"messages": [llm_with_tools.invoke(state["messages"])]}

def tools_exec_node(state: AgentState) -> dict:
    """Execute every tool call the LLM requested."""
    last_msg = state["messages"][-1]
    results  = []
    for tc in last_msg.tool_calls:
        output = tool_map[tc["name"]].invoke(tc["args"])
        results.append(ToolMessage(content=str(output), tool_call_id=tc["id"]))
    return {"messages": results}

# ── Routing: loop or stop ─────────────────────────────────────────────────
def should_continue(state: AgentState) -> Literal["tools", "__end__"]:
    """If the last message has tool calls -> tools; otherwise -> END."""
    last = state["messages"][-1]
    if hasattr(last, "tool_calls") and last.tool_calls:
        return "tools"
    return END

# ── Build cyclic graph ────────────────────────────────────────────────────
agent_builder = StateGraph(AgentState)
agent_builder.add_node("agent", agent_llm_node)
agent_builder.add_node("tools", tools_exec_node)

agent_builder.add_edge(START, "agent")
agent_builder.add_conditional_edges("agent", should_continue)
agent_builder.add_edge("tools", "agent")        # \u2190 CYCLE: tools feeds back to agent

agent_graph = agent_builder.compile()

print("Graph: START \u2192 agent \u2192 (tool_calls?) \u2192 tools \u2192 agent \u2026 OR \u2192 END\n")

for query in [
    "What is 47 plus 89?",
    "What is today's date and the square root of 225?",
]:
    print("\u2500" * 52)
    print(f"\U0001f464 Query : {query}")
    r = agent_graph.invoke({"messages": [HumanMessage(content=query)]})
    print(f"\U0001f916 Answer: {r['messages'][-1].content.strip()}")
    print(f"   Trace : {len(r['messages'])} messages exchanged")
    print()


In [ ]:
# ── Persistent Memory with MemorySaver + thread_id ───────────────────────
print("\U0001f9e0 Memory / Checkpoints with MemorySaver")
print("=" * 55)

class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage], operator.add]

def chat_node(state: ChatState) -> dict:
    return {"messages": [chat_model.invoke(state["messages"])]}

chat_builder = StateGraph(ChatState)
chat_builder.add_node("chat", chat_node)
chat_builder.add_edge(START, "chat")
chat_builder.add_edge("chat", END)

# ── Attach checkpointer ───────────────────────────────────────────────────
mem_saver     = MemorySaver()
persistent_chat = chat_builder.compile(checkpointer=mem_saver)

# ── Alice's session ───────────────────────────────────────────────────────
alice_cfg  = {"configurable": {"thread_id": "alice-001"}}
alice_turns = [
    "Hi! My name is Alice and I am a cloud architect at a fintech firm.",
    "I mainly use AWS and Kubernetes day to day.",
    "What do you know about me so far?",    # tests memory recall
]

print("\u2500\u2500 Alice's session (thread_id = alice-001) \u2500\u2500")
for msg in alice_turns:
    r = persistent_chat.invoke({"messages": [HumanMessage(content=msg)]}, config=alice_cfg)
    reply = r["messages"][-1].content.strip()
    print(f"\U0001f464 Alice : {msg}")
    print(f"\U0001f916 AI    : {reply[:210]}")
    print()

# ── Bob's session (different thread_id \u2192 fresh memory) ────────────────────
bob_cfg = {"configurable": {"thread_id": "bob-001"}}
r_bob   = persistent_chat.invoke(
    {"messages": [HumanMessage(content="What is my name?")]}, config=bob_cfg
)
print("\u2500\u2500 Bob's session (thread_id = bob-001 \u2014 fresh memory) \u2500\u2500")
print(f"\U0001f464 Bob : What is my name?")
print(f"\U0001f916 AI  : {r_bob['messages'][-1].content.strip()[:210]}")


In [ ]:
# ── Human-in-the-Loop (HITL) with interrupt_before ───────────────────────
print("\U0001f64b Human-in-the-Loop (HITL)")
print("=" * 55)
print()

# Compile agent_builder with interrupt_before + a fresh checkpointer
hitl_ckpt  = MemorySaver()
hitl_graph = agent_builder.compile(
    checkpointer=hitl_ckpt,
    interrupt_before=["tools"],      # \u2190 pause BEFORE the tools node
)

# ── Demo 1: Human APPROVES the tool call ─────────────────────────────────
cfg1   = {"configurable": {"thread_id": "hitl-approve"}}
query1 = "What is the square root of 625?"

print("\u2500" * 52)
print(f"\U0001f464 User   : {query1}")
print("\u2500" * 52)

# Run until the interrupt fires
paused = hitl_graph.invoke(
    {"messages": [HumanMessage(content=query1)]},
    config=cfg1,
)

last_msg = paused["messages"][-1]
print("\n\u23f8\ufe0f  PAUSED \u2014 waiting for human review of proposed tool call(s):\n")
if hasattr(last_msg, "tool_calls") and last_msg.tool_calls:
    for tc in last_msg.tool_calls:
        print(f"     Tool : {tc['name']}")
        print(f"     Args : {tc['args']}")

print("\n\u2705 Human decision: APPROVED \u2014 resuming execution...")
final = hitl_graph.invoke(None, config=cfg1)    # pass None to resume
print(f"\U0001f916 Final : {final['messages'][-1].content.strip()}")

# ── Demo 2: Human REJECTS the tool call ──────────────────────────────────
print()
print("\u2500" * 52)
cfg2   = {"configurable": {"thread_id": "hitl-reject"}}
query2 = "What is 100 plus 200?"

print(f"\U0001f464 User   : {query2}")
print("\u2500" * 52)

paused2 = hitl_graph.invoke(
    {"messages": [HumanMessage(content=query2)]},
    config=cfg2,
)

last2 = paused2["messages"][-1]
print("\n\u23f8\ufe0f  PAUSED \u2014 proposed tool call(s):")
if hasattr(last2, "tool_calls") and last2.tool_calls:
    for tc in last2.tool_calls:
        print(f"   Tool: {tc['name']}  Args: {tc['args']}")

print("\n\u274c Human decision: REJECTED \u2014 NOT resuming")
print("   Execution stops here. The paused state remains in the checkpointer.")
print("   (In production: log the rejection, notify the user, or update state before declining.)")


In [ ]:
# ── Human-in-the-Loop (HITL) with USER INPUT for approval/rejection ──────
print("🙋 Human-in-the-Loop (HITL) - Interactive User Decision")
print("=" * 55)
print()

# Reset with fresh checkpointer
hitl_ckpt_interactive = MemorySaver()
hitl_graph_interactive = agent_builder.compile(
    checkpointer=hitl_ckpt_interactive,
    interrupt_before=["tools"],
)

# Function to handle user approval/rejection
def hitl_interactive_demo(user_query: str, thread_id: str):
    """
    Run a single HITL demo with user input for approval/rejection.
    
    Args:
        user_query: The question to ask the agent
        thread_id:  Unique identifier for this session
    """
    cfg = {"configurable": {"thread_id": thread_id}}
    
    print("─" * 52)
    print(f"👤 User   : {user_query}")
    print("─" * 52)
    
    # Run until the interrupt fires
    paused = hitl_graph_interactive.invoke(
        {"messages": [HumanMessage(content=user_query)]},
        config=cfg,
    )
    
    last_msg = paused["messages"][-1]
    print("\n⏸️  PAUSED — proposed tool call(s):\n")
    if hasattr(last_msg, "tool_calls") and last_msg.tool_calls:
        for tc in last_msg.tool_calls:
            print(f"     Tool : {tc['name']}")
            print(f"     Args : {tc['args']}")
    else:
        print("   (No tool calls)")
    
    # ASK THE USER FOR INPUT
    print()
    decision = input("❓ Do you approve this tool call? (approved/rejected): ").strip().lower()
    
    if decision == "approved":
        print("\n✅ Decision: APPROVED — resuming execution...")
        final = hitl_graph_interactive.invoke(None, config=cfg)
        print(f"🤖 Final : {final['messages'][-1].content.strip()}")
    elif decision == "rejected":
        print("\n❌ Decision: REJECTED — NOT resuming")
        print("   Execution stops here. The paused state remains in the checkpointer.")
    else:
        print(f"\n⚠️  Invalid input: '{decision}'. Please enter 'approved' or 'rejected'.")
    
    print()

# ── Run interactive demos ─────────────────────────────────────────────────
hitl_interactive_demo("What is the square root of 625?", "hitl-interactive-1")
hitl_interactive_demo("What is 100 plus 200?", "hitl-interactive-2")


In [ ]:
# ── Human-in-the-Loop Loan Approval System  ─────────────────────
print("🏦 Loan Approval System with Human Review")
print("=" * 60)
print()
@tool
def check_credit_score(applicant_id: str) -> str:
    """Check the applicant's credit score."""
    scores = {
        "APP-001": {"score": 750, "status": "Excellent"},
        "APP-002": {"score": 620, "status": "Fair"},
        "APP-003": {"score": 800, "status": "Excellent"},
    }
    data = scores.get(applicant_id, {"score": 700, "status": "Good"})
    return f"Credit Score: {data['score']} ({data['status']})"

@tool
def verify_income(applicant_id: str) -> str:
    """Verify annual income."""
    incomes = {
        "APP-001": 85000,
        "APP-002": 45000,
        "APP-003": 120000,
    }
    income = incomes.get(applicant_id, 70000)
    return f"Annual Income: ${income:,}"


@tool
def calculate_dti(applicant_id: str, loan_amount: float) -> str:
    """Calculate Debt-to-Income ratio using actual verified income."""
    
    # Get the actual verified income for THIS applicant
    incomes = {
        "APP-001": 85000,
        "APP-002": 45000,
        "APP-003": 120000,
    }
    
    annual_income = incomes.get(applicant_id, 70000)  # ✅ Use real income
    
    # Calculate monthly obligations
    monthly_payment = (loan_amount / 60) + 500  # 5-year loan + existing debts
    monthly_income = annual_income / 12
    
    # Calculate DTI ratio
    dti_ratio = (monthly_payment / monthly_income) * 100
    
    status = "✅ PASS" if dti_ratio < 43 else "❌ FAIL"
    return f"Debt-to-Income Ratio: {dti_ratio:.1f}% {status} (max: 43%)"

@tool
def verify_employment(applicant_id: str) -> str:
    """Verify employment status."""
    employments = {
        "APP-001": "Employed - Tech Corp (5 years)",
        "APP-002": "Self-employed - Freelancer (2 years)",
        "APP-003": "Employed - Finance Co (8 years)",
    }
    return employments.get(applicant_id, "Employed (3 years)")

    
# ── Create Loan Approval Agent ───────────────────────────────────────────
loan_tools = [check_credit_score, verify_income, calculate_dti, verify_employment]
tool_map_loans = {t.name: t for t in loan_tools}
llm_loans = chat_model.bind_tools(loan_tools)
class LoanState(TypedDict):
    applicant_id: str
    loan_amount: float
    messages: Annotated[list[BaseMessage], operator.add]

def loan_agent(state: LoanState) -> dict:
    """Agent gathers loan info and makes a COMPREHENSIVE recommendation."""
    prompt = f"""You are a loan approval specialist. Analyze this loan application:
    - Applicant: {state['applicant_id']}
    - Loan Amount: ${state['loan_amount']:,.0f}
    
    Use ALL available tools to verify:
    1. Credit score (CIBIL/credit score)
    2. Annual income
    3. Debt-to-income ratio
    4. Employment status
    
    After gathering all information, provide a CLEAR RECOMMENDATION:
    
    Format your response like this:
    ════════════════════════════════════════════════════════
    RECOMMENDATION: [APPROVE / REJECT]
    
    Approval Factors:
    ✅ Credit Score: [score and status]
    ✅ Annual Income: [amount and assessment]
    ✅ Debt-to-Income Ratio: [percentage and status]
    ✅ Employment: [status and stability]
    
    Risk Assessment: [LOW / MEDIUM / HIGH]
    
    Reasoning:
    [Explain your decision clearly]
    ════════════════════════════════════════════════════════
    
    Be specific about WHY you approve or reject."""
    
    return {"messages": [llm_loans.invoke(state["messages"] + [HumanMessage(content=prompt)])]}

def loan_tools_exec(state: LoanState) -> dict:
    """Execute verification tools."""
    last_msg = state["messages"][-1]
    results = []
    for tc in last_msg.tool_calls:
        output = tool_map_loans[tc["name"]].invoke(tc["args"])
        results.append(ToolMessage(content=output, tool_call_id=tc["id"]))
    return {"messages": results}

def should_continue_loan(state: LoanState) -> Literal["loan_tools", "__end__"]:
    """Route: if tools needed, go to tools; else end."""
    last = state["messages"][-1]
    if hasattr(last, "tool_calls") and last.tool_calls:
        return "loan_tools"
    return END

# Build loan approval graph
loan_builder = StateGraph(LoanState)
loan_builder.add_node("loan_agent", loan_agent)
loan_builder.add_node("loan_tools", loan_tools_exec)
loan_builder.add_edge(START, "loan_agent")
loan_builder.add_conditional_edges("loan_agent", should_continue_loan)
loan_builder.add_edge("loan_tools", "loan_agent")

# Pause BEFORE gathering sensitive financial info
loan_hitl = loan_builder.compile(
    checkpointer=MemorySaver(),
    interrupt_before=["loan_tools"],
)

# ── Helper function to extract final answer ───────────────────────────────
def get_final_answer(result):
    """Extract the final answer from messages."""
    for msg in reversed(result["messages"]):
        if hasattr(msg, "content") and isinstance(msg.content, str):
            if msg.content.strip():
                return msg.content.strip()
    return "No recommendation found"


# ── Interactive Loan Approval Function ───────────────────────────────────
def process_loan_application(applicant_id: str, loan_amount: float):
    """Process loan with human approval at each step."""
    cfg = {"configurable": {"thread_id": f"loan-{applicant_id}"}}
    
    print("\n" + "─" * 60)
    print(f"📋 Application: {applicant_id}")
    print(f"💰 Loan Amount: ${loan_amount:,.0f}")
    print("─" * 60)
    
    print("\n🤖 Agent: Preparing to verify application...")
    
    paused = loan_hitl.invoke(
        {
            "applicant_id": applicant_id,
            "loan_amount": loan_amount,
            "messages": [HumanMessage(content="Process loan application")],
        },
        config=cfg,
    )
    
    last_msg = paused["messages"][-1]
    
    print("\n⏸️  PAUSED - Will verify:")
    if hasattr(last_msg, "tool_calls") and last_msg.tool_calls:
        for tc in last_msg.tool_calls:
            print(f"   • {tc['name']}")
    
    decision = input("\n👤 Loan Officer: Proceed with verification? (approved/rejected): ").strip().lower()
    
    if decision != "approved":
        print("\n❌ Application stopped by loan officer.")
        return
    
    print("\n✅ Verified! Gathering information...\n")
    
    final = loan_hitl.invoke(None, config=cfg)
    
    recommendation = get_final_answer(final)
    
    print("\n" + "─" * 60)
    print("🤖 AGENT RECOMMENDATION:")
    print("─" * 60)
    print(recommendation)
    
    print("\n" + "─" * 60)
    final_decision = input("👤 Loan Officer: Final decision? (approved/rejected): ").strip().lower()
    
    if final_decision == "approved":
        print(f"\n✅ LOAN APPROVED - {applicant_id}")
        print(f"   Amount: ${loan_amount:,.0f}")
    else:
        print(f"\n❌ LOAN REJECTED - {applicant_id}")
        print(f"   Amount: ${loan_amount:,.0f}")
    
    print("─" * 60 + "\n")

# ── RUN EXAMPLES ──────────────────────────────────────────────────────────
process_loan_application("APP-001", 250000)
process_loan_application("APP-002", 150000)
